In [1]:
"""
PIPELINE STAGE: Multi-Source Energy–Population Data Integration, Standardization & Feature Harmonization

INPUT:
    processed_data/steps/07_Capacity_Generation.xlsx
    processed_data/steps/08_Population.xlsx

OUTPUT:
    processed_data/steps/09_Capacity_Generation_Population.xlsx

LIBRARIES: pandas, os

1. OBJECTIVE:
    Integrate energy system data (capacity + generation) with population data to
    construct a unified, analysis-ready panel dataset. The pipeline aligns spatial
    and temporal dimensions, standardizes feature naming conventions into English,
    removes non-informative records, and prepares the dataset for per-capita
    energy analysis and advanced statistical modeling.

2. DATA INGESTION:
    A. Capacity–Generation Dataset:
       Load merged dataset containing:
           - Installed capacity by energy source
           - Electricity generation by energy source
           - Spatial identifiers (Plaka, İLLER)
           - Temporal identifiers (Yıl, Ay)

    B. Population Dataset:
       Load provincial population data:
           - Plate (Plaka)
           - Year (Yıl)
           - Population (Nüfus)

3. DATA CLEANING & NORMALIZATION:
    A. Column Sanitization:
       Strip whitespace from all column names in both datasets to ensure
       consistent key matching.

    B. Structural Alignment:
       Ensure both datasets share compatible keys:

           - Plaka (Plate code)
           - Yıl (Year)

4. DATA MERGING:
    A. Join Strategy:
       Perform a left join:

           capacity_data LEFT JOIN population_data

       using:

           ["Plaka", "Yıl"]

    B. Purpose:
       Preserve all energy system records while enriching them with population data.

5. FEATURE STANDARDIZATION (ENGLISH NORMALIZATION):
    A. Column Renaming:
       Convert all key variables into standardized English schema:

           Spatial Fields:
               Plaka   → Plate
               İLLER   → Province
               Yıl     → Year
               Ay      → Month

           Population:
               Nüfus / Nifus → Population

           Energy Capacity:
               capacity_Biyokütle → capacity_Biomass
               capacity_Güneş     → capacity_Solar
               capacity_Hidrolik  → capacity_Hydropower
               capacity_Rüzgar    → capacity_Wind

           Energy Generation:
               generation_Biyokütle → generation_Biomass
               generation_Güneş     → generation_Solar
               generation_Hidrolik  → generation_Hydropower
               generation_Rüzgar    → generation_Wind

    B. Goal:
       Ensure international publication compatibility (Peer-reviewed standards).

6. DATA FILTERING (ZERO-CAPACITY REMOVAL):
    A. Target Columns:
       Define key capacity features:

           Biomass, Solar, Hydropower, Wind

    B. Filtering Rule:
       Remove rows where ALL capacity values are zero:

           keep row if at least one capacity > 0

    C. Purpose:
       Eliminate non-productive or structurally irrelevant observations.

7. REDUNDANCY HANDLING:
    A. Province Column Removal:
       If "İl" column exists, remove it to avoid duplication with "Province".

    B. Rationale:
       Prevent ambiguity between Turkish and English spatial identifiers.

8. DATA QUALITY ASSURANCE:
    A. Consistency Check:
       Ensure:
           - No duplicate spatial identifiers
           - Valid population assignment per Plate-Year pair

    B. Missing Population Handling:
       Population values may be NaN if unmatched; retained for completeness.

9. FINAL DATA STRUCTURE:
    The resulting dataset includes:

        - Plate (spatial identifier)
        - Province
        - Year, Month
        - Population
        - Installed capacity (by source)
        - Electricity generation (by source)

10. OUTPUT GENERATION:
    A. Export File:

           processed_data/steps/09_Capacity_Generation_Population.xlsx

    B. Export Format:
       - Excel (.xlsx)
       - No index column

11. PROCESS LOGGING & MONITORING:
    A. Runtime Outputs:
       - Row count after filtering
       - Final column list
       - Sample preview (first 5 rows)
       - Output file path confirmation

12. EXPECTED OUTPUT DATASET:
    This dataset forms a complete multi-dimensional energy–demography panel:

        - Spatial dimension: Province / Plate
        - Temporal dimension: Year / Month
        - Energy system: Capacity & Generation by source
        - Socio-economic layer: Population

    It is suitable for:
        - Per-capita energy analysis
        - Sustainability indicators (CEEI, emissions per capita)
        - Machine learning forecasting models
        - Spatial econometric analysis
        - Energy transition research


"""
import pandas as pd
import os

# =====================================================
# DOSYA YOLLARI
# =====================================================

base_dir = r"C:\Users\w11\dergi2"

capacity_file = os.path.join(
    base_dir,
    "processed_data",
    "steps",
    "07_capacity_generation.xlsx"
)

population_file = os.path.join(
    base_dir,
    "processed_data",
    "steps",
    "08_population.xlsx"
)

output_file = os.path.join(
    base_dir,
    "processed_data",
    "steps",
    "09_capacity_generation_population.xlsx"
)

# =====================================================
# DOSYALARI OKU
# =====================================================

df_capacity = pd.read_excel(capacity_file)
df_population = pd.read_excel(population_file)

# =====================================================
# SÜTUN İSİMLERİNİ TEMİZLE
# =====================================================

df_capacity.columns = df_capacity.columns.str.strip()
df_population.columns = df_population.columns.str.strip()

# =====================================================
# BİRLEŞTİR
# =====================================================

df_final = df_capacity.merge(
    df_population,
    on=["Plaka", "Yıl"],
    how="left"
)

# =====================================================
# SÜTUN ADLARINI İNGİLİZCEYE ÇEVİR VE DEĞİŞTİR
# (Hem temel sütunlar hem de kaynak isimleri)
# =====================================================

# Nüfus sütunu popülasyon tablosunda "Nüfus" veya "Nifus" olarak gelebilir, ikisini de kapsadık.
df_final = df_final.rename(columns={
    "Plaka": "Plate",
    "İLLER": "Province",
    "Yıl": "Year",
    "Ay": "Month",
    "Nüfus": "Population", 
    "Nifus": "Population",
    "capacity_Biyokütle": "capacity_Biomass",
    "capacity_Güneş": "capacity_Solar",
    "capacity_Hidrolik": "capacity_Hydropower",
    "capacity_Rüzgar": "capacity_Wind",
    "generation_Biyokütle": "generation_Biomass",
    "generation_Güneş": "generation_Solar",
    "generation_Hidrolik": "generation_Hydropower",
    "generation_Rüzgar": "generation_Wind"
})

# =====================================================
# SIFIR KAPASİTE FİLTRESİ
# =====================================================

# Tüm kapasite sütunlarını belirle
capacity_columns = [
    "capacity_Biomass", 
    "capacity_Solar", 
    "capacity_Hydropower", 
    "capacity_Wind"
]

# Belirlenen kaynakların HİÇBİRİNDE kurulu gücü olmayan (hepsi 0 olan) satırları sil,
# en az birinde kapasite varsa o satırı tut.
df_final = df_final[(df_final[capacity_columns] != 0).any(axis=1)]

# =====================================================
# GEREKSİZ SÜTUNLARI SİL (İl Sütunu)
# =====================================================

# Eğer "İl" sütunu veri setinde mevcutsa sil
if 'İl' in df_final.columns:
    df_final = df_final.drop(columns=['İl'])
    print(">>> 'İl' sütunu (tekrar içerdiği için) silindi.")

# =====================================================
# KAYDET VE KONTROL
# =====================================================

df_final.to_excel(output_file, index=False)

print("Tamamlandı.")
print("Satır Sayısı:", len(df_final))
print("Dosya:", output_file)

print("\nSütunlar:")
print(df_final.columns.tolist())

print("\nİlk 5 Satır:")
print(df_final.head())

>>> 'İl' sütunu (tekrar içerdiği için) silindi.
Tamamlandı.
Satır Sayısı: 4790
Dosya: C:\Users\w11\dergi2\processed_data\steps\09_capacity_generation_population.xlsx

Sütunlar:
['Plate', 'Province', 'capacity_Biomass', 'capacity_Solar', 'capacity_Hydropower', 'capacity_Wind', 'Year', 'Month', 'generation_Biomass', 'generation_Solar', 'generation_Hydropower', 'generation_Wind', 'Population']

İlk 5 Satır:
   Plate Province  capacity_Biomass  capacity_Solar  capacity_Hydropower  \
0      1    ADANA              2.13          132.69                  0.0   
1      1    ADANA              2.13          133.68                  0.0   
2      1    ADANA              2.13          133.98                  0.0   
3      1    ADANA              2.13          134.48                  0.0   
4      1    ADANA              2.13          134.48                  0.0   

   capacity_Wind  Year  Month  generation_Biomass  generation_Solar  \
0            0.0  2021      1              100.35          11724